In [ ]:
# Import all required libraries
import sys
import torch
import os
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics.pairwise import rbf_kernel
from torch_geometric.utils import k_hop_subgraph
from tqdm import tqdm
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

print("[OK] All libraries imported")

✓ All libraries imported


In [ ]:
# ============================================================================
# MODEL ARCHITECTURE (Same as Day 1)
# ============================================================================

class AnomalyGCN(torch.nn.Module):
    """
    Advanced GCN for Anomaly Detection with:
    - Encoder-decoder architecture
    - Skip connections
    - Feature reconstruction
    - Contrastive learning support
    """
    def __init__(self, in_channels, hidden_channels=64):
        super(AnomalyGCN, self).__init__()
        
        # Encoder
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.ln1 = torch.nn.LayerNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.ln2 = torch.nn.LayerNorm(hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels // 2)
        self.ln3 = torch.nn.LayerNorm(hidden_channels // 2)
        
        # Decoder (for reconstruction)
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels // 2, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(hidden_channels, in_channels)
        )
        
        # Anomaly score head
        self.score_head = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels // 2, hidden_channels // 4),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(hidden_channels // 4, 1)
        )
        
        self._reset_parameters()
        
    def _reset_parameters(self):
        for conv in [self.conv1, self.conv2, self.conv3]:
            if hasattr(conv, 'reset_parameters'):
                conv.reset_parameters()
        for module in [self.decoder, self.score_head]:
            for layer in module:
                if hasattr(layer, 'reset_parameters'):
                    layer.reset_parameters()
    
    def encode(self, x, edge_index):
        # Encoder path with skip connections
        x1 = self.conv1(x, edge_index)
        x1 = self.ln1(x1)
        x1 = F.relu(x1)
        x1 = F.dropout(x1, p=0.2, training=self.training)
        
        x2 = self.conv2(x1, edge_index)
        x2 = self.ln2(x2)
        x2 = F.relu(x2)
        x2 = F.dropout(x2, p=0.2, training=self.training)
        x2 = x2 + x1  # Skip connection
        
        x3 = self.conv3(x2, edge_index)
        x3 = self.ln3(x3)
        x3 = F.relu(x3)
        
        return x3
    
    def forward(self, x, edge_index, return_embedding=False):
        # Encode
        embedding = self.encode(x, edge_index)
        
        # Reconstruct features
        x_recon = self.decoder(embedding)
        
        # Anomaly scores
        scores = self.score_head(embedding).squeeze()
        
        if return_embedding:
            return scores, x_recon, embedding
        return scores, x_recon

print("[OK] AnomalyGCN model class defined")

✓ AnomalyGCN model class defined


In [ ]:
# ============================================================================
# DAY 2: GRAPHLIME EXPLANATIONS - SETUP
# ============================================================================

print("="*70)
print("DAY 2: GRAPHLIME FEATURE ATTRIBUTION")
print("="*70)

# Load Day 1 results
print("\n[1/4] Loading Day 1 results...")

# Check if results file exists
if not os.path.exists('../results/anomaly_detection_results.pt'):
    print("[ERROR] anomaly_detection_results.pt not found!")
    print("Please run Day 1 training first.")
    raise FileNotFoundError("Run Day 1 notebook first to generate results")
else:
    # Load with weights_only=False since file contains numpy arrays and PyG Data objects
    results = torch.load('../results/anomaly_detection_results.pt', weights_only=False)
    
    model_state = results['model_state_dict']
    combined_scores = results['combined_scores']  # This is the actual anomaly score used
    anomalies = results['anomalies']
    node_ids = results['node_ids']
    data = results['data']
    
    print(f"[OK] Loaded Day 1 results")
    print(f"  Total nodes: {len(node_ids):,}")
    print(f"  Anomalies found: {anomalies.sum():,} ({100*anomalies.float().mean():.2f}%)")
    print(f"  Combined score range: [{combined_scores.min():.2f}, {combined_scores.max():.2f}]")
    print(f"  Combined score std: {combined_scores.std():.2f}")

# Reconstruct the model (use the same architecture from Day 1)
print("\n[2/4] Reconstructing model...")
device = 'cpu'

# Use the same AnomalyGCN model from Day 1
model = AnomalyGCN(data.num_node_features, hidden_channels=64).to(device)
model.load_state_dict(model_state)
model.eval()

print(f"[OK] Model reconstructed and set to eval mode")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Select nodes to explain
print("\n[3/4] Selecting nodes to explain...")

# Get indices of anomalous nodes
anomaly_indices = np.where(anomalies)[0]

# Convert combined_scores to numpy for easier indexing
combined_scores_np = combined_scores.cpu().numpy()

# Sort by anomaly score (most anomalous = highest combined scores)
sorted_indices = anomaly_indices[np.argsort(combined_scores_np[anomaly_indices])[::-1]]

# Select top N most anomalous
TOP_N = 50  # Start with 50, you can increase later

selected_indices = sorted_indices[:TOP_N]

print(f"  Total anomalies: {len(anomaly_indices):,}")
print(f"  Explaining top: {len(selected_indices)}")
print(f"  Score range of selected: [{combined_scores_np[selected_indices].min():.2f}, {combined_scores_np[selected_indices].max():.2f}]")

print("\n[4/4] Ready to generate explanations!")
print("="*70)

DAY 2: GRAPHLIME FEATURE ATTRIBUTION

[1/4] Loading Day 1 results...
✓ Loaded Day 1 results
  Total nodes: 203,769
  Anomalies found: 10,189 (5.00%)
  Combined score range: [0.00, 0.81]
  Combined score std: 0.03

[2/4] Reconstructing model...
✓ Model reconstructed and set to eval mode
  Parameters: 32,888

[3/4] Selecting nodes to explain...
  Total anomalies: 10,189
  Explaining top: 50
  Score range of selected: [0.41, 0.81]

[4/4] Ready to generate explanations!


In [ ]:
# ============================================================================
# DAY 2: GRAPHLIME CLASS IMPLEMENTATION
# ============================================================================

class GraphLIME:
    """
    GraphLIME: Local Interpretable Model Explanations for GNN
    
    This explains individual node predictions by:
    1. Extracting a local subgraph around the target node
    2. Fitting a simple linear model to approximate GNN behavior locally
    3. Using feature weights from linear model as explanations
    """
    
    def __init__(self, model, hop=2, rho=0.1):
        """
        Args:
            model: Trained GNN model
            hop: Number of hops for subgraph (2 = neighbors + neighbors of neighbors)
            rho: Regularization strength for Lasso
        """
        self.model = model
        self.hop = hop
        self.rho = rho
        
    def explain_node(self, node_idx, data, device='cpu'):
        """
        Generate explanation for a single node
        
        Args:
            node_idx: Index of node to explain (can be int, numpy int, or tensor)
            data: PyG Data object
            device: 'cpu' or 'cuda'
            
        Returns:
            feature_weights: Numpy array of feature importance weights
        """
        # Convert node_idx to int (in case it's numpy.int64 or tensor)
        if isinstance(node_idx, (np.integer, np.ndarray)):
            node_idx = int(node_idx)
        elif torch.is_tensor(node_idx):
            node_idx = node_idx.item()
        
        # Step 1: Extract k-hop subgraph around target node
        subset, edge_index, mapping, edge_mask = k_hop_subgraph(
            node_idx, 
            self.hop, 
            data.edge_index,
            num_nodes=data.num_nodes
        )
        
        # Handle edge case: single node with no neighbors
        if len(subset) == 1:
            # For isolated nodes, return uniform feature importance
            return np.ones(data.num_node_features) / data.num_node_features
        
        # Limit subgraph size for computational efficiency
        max_nodes = 3000 
        if len(subset) > max_nodes:
            # Keep closest nodes (first k nodes are typically closest)
            subset = subset[:max_nodes]
            # Recompute edges for reduced subset
            subset_set = set(subset.tolist())
            edge_mask = torch.tensor([
                (data.edge_index[0, i].item() in subset_set and 
                 data.edge_index[1, i].item() in subset_set)
                for i in range(data.edge_index.shape[1])
            ])
            edge_index = data.edge_index[:, edge_mask]
        
        # Step 2: Get features for subgraph nodes
        sub_x = data.x[subset].cpu().numpy()
        
        # Ensure sub_x is 2D
        if sub_x.ndim == 1:
            sub_x = sub_x.reshape(1, -1)
        
        # Use RobustScaler (same as Day 1 preprocessing)
        # This is more robust to outliers than StandardScaler
        scaler = RobustScaler()
        try:
            sub_x_scaled = scaler.fit_transform(sub_x)
        except ValueError:
            # If scaling fails (e.g., single sample), use original features
            sub_x_scaled = sub_x
        
        # Step 3: Get GNN predictions for subgraph nodes
        self.model.eval()
        with torch.no_grad():
            # Create subgraph edge index (remap to local indices)
            # subset.tolist() already returns Python ints, no need for .item()
            idx_map = {old: new for new, old in enumerate(subset.tolist())}
            
            remapped_edges = []
            for src, dst in edge_index.t():
                src_item = src.item()
                dst_item = dst.item()
                if src_item in idx_map and dst_item in idx_map:
                    remapped_edges.append([idx_map[src_item], idx_map[dst_item]])
            
            if len(remapped_edges) > 0:
                sub_edge_index = torch.LongTensor(remapped_edges).t()
            else:
                # No edges - create empty edge index
                sub_edge_index = torch.LongTensor([[], []]).long()
            
            # Get GNN predictions for subgraph
            sub_x_tensor = torch.FloatTensor(sub_x).to(device)
            sub_edge_index = sub_edge_index.to(device)
            
            # Model returns (scores, x_recon) - we only need scores
            sub_scores, _ = self.model(sub_x_tensor, sub_edge_index)
            sub_scores = sub_scores.cpu().numpy()
            
            # Handle scalar output
            if sub_scores.ndim == 0:
                sub_scores = np.array([sub_scores])
            elif sub_scores.ndim == 1 and len(sub_scores) == 1:
                # Single prediction, but we need at least 2 for linear model
                # Return uniform feature importance
                return np.ones(data.num_node_features) / data.num_node_features
        
        # Step 4: Compute similarity weights using RBF kernel
        # Nodes closer to target node get higher weight
        target_idx_in_subgraph = mapping.item()
        target_features = sub_x_scaled[target_idx_in_subgraph].reshape(1, -1)
        
        # RBF kernel: exp(-gamma * ||x - target||^2)
        similarity = rbf_kernel(sub_x_scaled, target_features).flatten()
        
        # Ensure we have enough samples for linear model (at least 2)
        if len(sub_x_scaled) < 2:
            return np.ones(data.num_node_features) / data.num_node_features
        
        # Step 5: Fit local linear model
        # Use Lasso (L1 regularization) to get sparse explanations
        try:
            local_model = Lasso(alpha=self.rho, positive=True, max_iter=5000, random_state=42)
            local_model.fit(sub_x_scaled, sub_scores, sample_weight=similarity)
            feature_weights = local_model.coef_
        except Exception as e:
            # Fallback to Ridge if Lasso fails
            try:
                local_model = Ridge(alpha=self.rho, random_state=42)
                local_model.fit(sub_x_scaled, sub_scores, sample_weight=similarity)
                feature_weights = np.abs(local_model.coef_)
            except Exception:
                # If both fail, return uniform weights
                feature_weights = np.ones(data.num_node_features) / data.num_node_features
        
        return feature_weights
    
    def get_top_features(self, feature_weights, top_k=3, threshold=1e-8):
        """
        Extract top-k most important features
        
        Args:
            feature_weights: Array of feature importance weights
            top_k: Number of top features to return
            threshold: Minimum absolute weight to consider
            
        Returns:
            List of (feature_index, weight) tuples
        """
        # Find features above threshold
        important_idx = np.where(np.abs(feature_weights) > threshold)[0]
        
        if len(important_idx) == 0:
            # No features above threshold - just take top k by absolute value
            sorted_idx = np.argsort(np.abs(feature_weights))[::-1]
            top_idx = sorted_idx[:top_k]
            top_weights = feature_weights[top_idx]
            return list(zip(top_idx, top_weights))
        
        # Sort by absolute importance
        important_weights = feature_weights[important_idx]
        sorted_idx = np.argsort(np.abs(important_weights))[::-1]
        
        # Get top k
        top_idx = important_idx[sorted_idx[:top_k]]
        top_weights = important_weights[sorted_idx[:top_k]]
        
        return list(zip(top_idx, top_weights))

print("[OK] GraphLIME class implemented")
print("\nGraphLIME workflow:")
print("  1. Extract local subgraph (k-hop neighbors)")
print("  2. Get GNN predictions for subgraph")
print("  3. Fit linear model weighted by node similarity")
print("  4. Use linear model coefficients as feature importance")

✓ GraphLIME class implemented

GraphLIME workflow:
  1. Extract local subgraph (k-hop neighbors)
  2. Get GNN predictions for subgraph
  3. Fit linear model weighted by node similarity
  4. Use linear model coefficients as feature importance


In [ ]:
# ============================================================================
# DAY 2: GENERATE GRAPHLIME EXPLANATIONS
# ============================================================================

print("\n" + "="*70)
print("GENERATING EXPLANATIONS")
print("="*70)

# Initialize explainer
explainer = GraphLIME(model, hop=2, rho=0.1)

print(f"\nConfiguration:")
print(f"  Explainer: GraphLIME")
print(f"  K-hop: 2 (neighbors + neighbors of neighbors)")
print(f"  Regularization (rho): 0.1")
print(f"  Top features per node: 3")
print(f"  Nodes to explain: {len(selected_indices)}")
print(f"\n⏰ Estimated time: {len(selected_indices) * 1.0:.0f}-{len(selected_indices) * 1.5:.0f} seconds")
print(f"   (~{len(selected_indices) * 1.0 / 60:.1f}-{len(selected_indices) * 1.5 / 60:.1f} minutes)\n")

# Generate explanations
explanations = {}
failed_nodes = []
start_time = time.time()

print("Starting explanation generation...\n")

for i, idx in enumerate(tqdm(selected_indices, desc="Explaining nodes", ncols=100)):
    try:
        # Get feature importance weights from GraphLIME
        weights = explainer.explain_node(idx, data, device='cpu')
        
        # Get top 3 most important features
        top_features = explainer.get_top_features(weights, top_k=3)
        
        # Store explanation with all relevant info
        explanations[int(idx)] = {
            'node_id': node_ids[idx],
            'anomaly_score': float(combined_scores_np[idx]),  # Use combined_scores
            'feature_weights': weights,  # Full weight vector
            'top_features': top_features,  # Top 3: [(idx, weight), ...]
            'node_features': data.x[idx].cpu().numpy()  # Actual feature values
        }
        
        # Progress update every 10 nodes
        if (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / (i + 1)
            eta = avg_time * (len(selected_indices) - i - 1)
            success_rate = len(explanations) / (i + 1) * 100
            
            print(f"\n  Progress: {i+1}/{len(selected_indices)} | "
                  f"Success: {success_rate:.1f}% | "
                  f"Elapsed: {elapsed:.0f}s | "
                  f"ETA: {eta:.0f}s | "
                  f"Failed: {len(failed_nodes)}")
        
    except Exception as e:
        failed_nodes.append(idx)
        # Only print first 3 errors to avoid spam
        if len(failed_nodes) <= 3:
            print(f"\n  ⚠️  Failed on node {idx}: {str(e)[:80]}")
        continue

elapsed_time = time.time() - start_time

print("\n" + "="*70)
print("EXPLANATION GENERATION COMPLETE")
print("="*70)

print(f"\nResults:")
print(f"  Total attempted: {len(selected_indices)}")
print(f"  Successful: {len(explanations)}")
print(f"  Failed: {len(failed_nodes)}")
print(f"  Success rate: {len(explanations) / len(selected_indices) * 100:.1f}%")
print(f"  Time taken: {elapsed_time:.1f} seconds ({elapsed_time / 60:.1f} minutes)")
print(f"  Avg time per node: {elapsed_time / len(selected_indices):.2f} seconds")

if len(failed_nodes) > 0:
    print(f"\n[WARNING] Failed node indices: {failed_nodes[:10]}")  # Show first 10
    if len(failed_nodes) > 10:
        print(f"     ... and {len(failed_nodes) - 10} more")


GENERATING EXPLANATIONS

Configuration:
  Explainer: GraphLIME
  K-hop: 2 (neighbors + neighbors of neighbors)
  Regularization (rho): 0.1
  Top features per node: 3
  Nodes to explain: 50

⏰ Estimated time: 50-75 seconds
   (~0.8-1.2 minutes)

Starting explanation generation...



Explaining nodes:  12%|█████▌                                        | 6/50 [00:00<00:00, 58.13it/s]


  Progress: 10/50 | Success: 100.0% | Elapsed: 0s | ETA: 1s | Failed: 0


Explaining nodes:  40%|██████████████████                           | 20/50 [00:00<00:00, 63.94it/s]


  Progress: 20/50 | Success: 100.0% | Elapsed: 0s | ETA: 0s | Failed: 0

  Progress: 30/50 | Success: 100.0% | Elapsed: 0s | ETA: 0s | Failed: 0


Explaining nodes:  72%|████████████████████████████████▍            | 36/50 [00:00<00:00, 97.32it/s]


  Progress: 40/50 | Success: 100.0% | Elapsed: 0s | ETA: 0s | Failed: 0


Explaining nodes: 100%|█████████████████████████████████████████████| 50/50 [00:00<00:00, 89.84it/s]


  Progress: 50/50 | Success: 100.0% | Elapsed: 1s | ETA: 0s | Failed: 0

EXPLANATION GENERATION COMPLETE

📊 Results:
  Total attempted: 50
  Successful: 50
  Failed: 0
  Success rate: 100.0%
  Time taken: 0.6 seconds (0.0 minutes)
  Avg time per node: 0.01 seconds


In [ ]:
# ============================================================================
# DAY 2: SAVE RESULTS AND SHOW EXAMPLES
# ============================================================================

print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

# Create results directory if needed
os.makedirs('../results', exist_ok=True)

# Save explanations
output_path = '../results/day2_explanations.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(explanations, f)

print(f"\n[SAVED] {len(explanations)} explanations to:")
print(f"  {output_path}")
print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Show sample explanations
print("\n" + "="*70)
print("SAMPLE EXPLANATIONS")
print("="*70)

if len(explanations) > 0:
    # Show 3 example explanations
    sample_count = min(3, len(explanations))
    sample_keys = list(explanations.keys())[:sample_count]
    
    for i, idx in enumerate(sample_keys, 1):
        explanation = explanations[idx]
        
        print(f"\n{'─'*70}")
        print(f"Example {i}:")
        print(f"{'─'*70}")
        print(f"Node ID: {explanation['node_id']}")
        print(f"Anomaly Score: {explanation['anomaly_score']:.4f} (higher = more suspicious)")
        print(f"\nTop 3 Most Important Features:")
        
        for j, (feat_idx, weight) in enumerate(explanation['top_features'], 1):
            feat_value = explanation['node_features'][feat_idx]
            print(f"  {j}. Feature {feat_idx:3d}:")
            print(f"     - GraphLIME importance: {weight:8.4f}")
            print(f"     - Actual value: {feat_value:8.4f}")
        
        print(f"\nInterpretation:")
        print(f"  The model flagged this wallet primarily because of:")
        for j, (feat_idx, weight) in enumerate(explanation['top_features'], 1):
            print(f"    • Feature {feat_idx} (importance: {abs(weight):.3f})")

# Summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

if len(explanations) > 0:
    # Collect all top features
    all_top_features = []
    for exp in explanations.values():
        all_top_features.extend([feat_idx for feat_idx, _ in exp['top_features']])
    
    # Count frequency
    from collections import Counter
    feature_counts = Counter(all_top_features)
    
    print(f"\nMost Frequently Important Features:")
    print(f"(These features are often cited as reasons for anomalies)")
    print()
    for feat_idx, count in feature_counts.most_common(10):
        pct = count / len(explanations) * 100
        print(f"  Feature {feat_idx:3d}: appears in {count:3d} explanations ({pct:5.1f}%)")

print("\n" + "="*70)
print("[COMPLETE] DAY 2 COMPLETE!")
print("="*70)

print(f"\nGenerated files:")
print(f"  [SAVED] ../results/day2_explanations.pkl")
print(f"\nWhat we learned:")
print(f"  • Explained {len(explanations)} anomalous wallets")
print(f"  • Identified which features matter most for fraud detection")
print(f"  • Created interpretable explanations for model decisions")
print(f"\nNext steps:")
print(f"  - Day 3: LLM Integration (convert to natural language)")
print(f"  - Day 4: Analysis & Visualization")
print(f"  - Day 5: Documentation & Presentation")

print("="*70)


SAVING RESULTS

✓ Saved 50 explanations to:
  ../results/day2_explanations.pkl
  File size: 95.1 KB

SAMPLE EXPLANATIONS

──────────────────────────────────────────────────────────────────────
Example 1:
──────────────────────────────────────────────────────────────────────
Node ID: 339576533
Anomaly Score: 0.8104 (higher = more suspicious)

Top 3 Most Important Features:
  1. Feature 182:
     - GraphLIME importance:   0.0055
     - Actual value:   2.9285
  2. Feature 181:
     - GraphLIME importance:   0.0055
     - Actual value:   9.1049
  3. Feature 180:
     - GraphLIME importance:   0.0055
     - Actual value:   7.5134

Interpretation:
  The model flagged this wallet primarily because of:
    • Feature 182 (importance: 0.005)
    • Feature 181 (importance: 0.005)
    • Feature 180 (importance: 0.005)

──────────────────────────────────────────────────────────────────────
Example 2:
──────────────────────────────────────────────────────────────────────
Node ID: 232446225
Anomaly 

In [ ]:
# ============================================================================
# IMPROVED GRAPHLIME V2: Enhanced Feature Diversity
# ============================================================================

class GraphLIME_V2:
    """
    Improved GraphLIME with better hyperparameters:
    - Lower regularization (rho=0.001 vs 0.1) for more feature diversity
    - Ridge regression (L2) instead of Lasso (L1) for gentler penalty
    - Higher top_k (10 vs 3) to see more important features
    """
    
    def __init__(self, model, hop=2, rho=0.001, use_ridge=True):
        self.model = model
        self.hop = hop
        self.rho = rho
        self.use_ridge = use_ridge
        
    def explain_node(self, node_idx, data, device='cpu'):
        """Generate explanation with improved settings"""
        # Convert node_idx to int
        if isinstance(node_idx, (np.integer, np.ndarray)):
            node_idx = int(node_idx)
        elif torch.is_tensor(node_idx):
            node_idx = node_idx.item()
        
        # Extract k-hop subgraph
        subset, edge_index, mapping, edge_mask = k_hop_subgraph(
            node_idx, self.hop, data.edge_index, num_nodes=data.num_nodes
        )
        
        # Handle edge case: single node
        if len(subset) == 1:
            return np.ones(data.num_node_features) / data.num_node_features
        
        # Limit subgraph size for efficiency
        max_nodes = 3000 
        if len(subset) > max_nodes:
            subset = subset[:max_nodes]
            subset_set = set(subset.tolist())
            edge_mask = torch.tensor([
                (data.edge_index[0, i].item() in subset_set and 
                 data.edge_index[1, i].item() in subset_set)
                for i in range(data.edge_index.shape[1])
            ])
            edge_index = data.edge_index[:, edge_mask]
        
        # Get and scale features
        sub_x = data.x[subset].cpu().numpy()
        if sub_x.ndim == 1:
            sub_x = sub_x.reshape(1, -1)
        
        scaler = RobustScaler()
        try:
            sub_x_scaled = scaler.fit_transform(sub_x)
        except ValueError:
            sub_x_scaled = sub_x
        
        # Get GNN predictions for subgraph
        self.model.eval()
        with torch.no_grad():
            idx_map = {old: new for new, old in enumerate(subset.tolist())}
            
            remapped_edges = []
            for src, dst in edge_index.t():
                src_item, dst_item = src.item(), dst.item()
                if src_item in idx_map and dst_item in idx_map:
                    remapped_edges.append([idx_map[src_item], idx_map[dst_item]])
            
            sub_edge_index = torch.LongTensor(remapped_edges).t() if remapped_edges else torch.LongTensor([[], []]).long()
            
            sub_x_tensor = torch.FloatTensor(sub_x).to(device)
            sub_edge_index = sub_edge_index.to(device)
            
            sub_scores, _ = self.model(sub_x_tensor, sub_edge_index)
            sub_scores = sub_scores.cpu().numpy()
            
            if sub_scores.ndim == 0:
                sub_scores = np.array([sub_scores])
            elif sub_scores.ndim == 1 and len(sub_scores) == 1:
                return np.ones(data.num_node_features) / data.num_node_features
        
        # Compute similarity weights (RBF kernel)
        target_idx_in_subgraph = mapping.item()
        target_features = sub_x_scaled[target_idx_in_subgraph].reshape(1, -1)
        similarity = rbf_kernel(sub_x_scaled, target_features).flatten()
        
        if len(sub_x_scaled) < 2:
            return np.ones(data.num_node_features) / data.num_node_features
        
        # Fit local linear model (Ridge for better feature diversity)
        try:
            if self.use_ridge:
                local_model = Ridge(alpha=self.rho, random_state=42)
            else:
                local_model = Lasso(alpha=self.rho, max_iter=5000, random_state=42)
            
            local_model.fit(sub_x_scaled, sub_scores, sample_weight=similarity)
            feature_weights = np.abs(local_model.coef_)
        except Exception:
            try:
                local_model = Ridge(alpha=0.01, random_state=42)
                local_model.fit(sub_x_scaled, sub_scores, sample_weight=similarity)
                feature_weights = np.abs(local_model.coef_)
            except Exception:
                feature_weights = np.ones(data.num_node_features) / data.num_node_features
        
        return feature_weights
    
    def get_top_features(self, feature_weights, top_k=10, threshold=1e-10):
        """Extract top-k most important features"""
        important_idx = np.where(np.abs(feature_weights) > threshold)[0]
        
        if len(important_idx) == 0:
            sorted_idx = np.argsort(np.abs(feature_weights))[::-1]
            return list(zip(sorted_idx[:top_k], feature_weights[sorted_idx[:top_k]]))
        
        important_weights = feature_weights[important_idx]
        sorted_idx = np.argsort(np.abs(important_weights))[::-1]
        top_idx = important_idx[sorted_idx[:top_k]]
        top_weights = important_weights[sorted_idx[:top_k]]
        
        return list(zip(top_idx, top_weights))

print("[OK] GraphLIME_V2 class defined")
print("  Key improvements: rho=0.001, Ridge regression, top_k=10")

✓ GraphLIME_V2 class defined
  Key improvements: rho=0.001, Ridge regression, top_k=10


In [ ]:
# ============================================================================
# GENERATE IMPROVED EXPLANATIONS (GRAPHLIME V2)
# ============================================================================

print("="*70)
print("GENERATING IMPROVED EXPLANATIONS WITH GRAPHLIME V2")
print("="*70)

# Initialize improved explainer
explainer_v2 = GraphLIME_V2(model, hop=2, rho=0.001, use_ridge=True)

print(f"\nExplaining {len(selected_indices)} anomalous nodes...")

# Generate explanations
explanations_v2 = {}
failed_v2 = []
start = time.time()

for idx in tqdm(selected_indices, desc="GraphLIME V2", ncols=100):
    try:
        weights = explainer_v2.explain_node(idx, data, device='cpu')
        top_features = explainer_v2.get_top_features(weights, top_k=10)
        
        explanations_v2[int(idx)] = {
            'node_id': node_ids[idx],
            'anomaly_score': float(combined_scores_np[idx]),
            'feature_weights': weights,
            'top_features': top_features,
            'node_features': data.x[idx].cpu().numpy()
        }
    except Exception as e:
        failed_v2.append((idx, str(e)))

elapsed = time.time() - start

# Results summary
print(f"\n{'='*70}")
print(f"[SUCCESS] {len(explanations_v2)}/{len(selected_indices)} nodes ({100*len(explanations_v2)/len(selected_indices):.1f}%)")
print(f"Time: {elapsed:.1f}s (avg: {elapsed/len(selected_indices):.2f}s per node)")
print(f"{'='*70}")

# Feature diversity comparison
all_v2_features = [f for exp in explanations_v2.values() for f, _ in exp['top_features']]
all_v1_features = [f for exp in explanations.values() for f, _ in exp['top_features']]

feature_counts_v2 = Counter(all_v2_features)
v2_unique = len(set(all_v2_features))
v1_unique = len(set(all_v1_features))

print(f"\nCOMPARISON:")
print(f"  Original GraphLIME: {v1_unique} unique features")
print(f"  Improved V2:        {v2_unique} unique features")
print(f"  Improvement:        {v2_unique - v1_unique} more features discovered ({v2_unique/v1_unique:.1f}x)")

print(f"\nTop 10 Most Important Features (V2):")
for feat, count in feature_counts_v2.most_common(10):
    pct = (count / len(all_v2_features)) * 100
    bar = "█" * int(pct / 2)
    print(f"  Feature {feat:>3}: {count:>3}x ({pct:>5.1f}%) {bar}")

# Save improved explanations
output_v2 = '../results/day2_explanations_v2.pkl'
with open(output_v2, 'wb') as f:
    pickle.dump(explanations_v2, f)

print(f"\n[SAVED] {output_v2} ({os.path.getsize(output_v2)/1024:.1f} KB)")

# Show examples
print(f"\n{'='*70}")
print("SAMPLE EXPLANATIONS (Top 3 Nodes)")
print(f"{'='*70}")

for i, idx in enumerate(list(explanations_v2.keys())[:3], 1):
    exp = explanations_v2[idx]
    print(f"\n[{i}] Node {exp['node_id']} | Anomaly Score: {exp['anomaly_score']:.4f}")
    print(f"    Top 5 Features:")
    for feat, weight in exp['top_features'][:5]:
        feat_val = exp['node_features'][feat]
        print(f"      • Feature {feat:>3}: importance={weight:.6f}, value={feat_val:.4f}")

print(f"\n{'='*70}")
print("[COMPLETE] DAY 2 COMPLETE - IMPROVED EXPLANATIONS GENERATED!")
print(f"{'='*70}")
print(f"\nOutput Files:")
print(f"  • Original:  ../results/day2_explanations.pkl    ({v1_unique} features)")
print(f"  • Improved:  ../results/day2_explanations_v2.pkl ({v2_unique} features)")
print(f"\nReady for Day 3: LLM Integration & Natural Language Explanations")

GENERATING IMPROVED EXPLANATIONS WITH GRAPHLIME V2

Explaining 50 anomalous nodes...


GraphLIME V2: 100%|████████████████████████████████████████████████| 50/50 [00:00<00:00, 107.24it/s]


✅ Success: 50/50 nodes (100.0%)
⏱️  Time: 0.5s (avg: 0.01s per node)

📊 COMPARISON:
  Original GraphLIME: 3 unique features
  Improved V2:        39 unique features
  🎯 Improvement:     36 more features discovered (13.0x)

🎯 Top 10 Most Important Features (V2):
  Feature 182:  47x (  9.4%) ████
  Feature 179:  47x (  9.4%) ████
  Feature 177:  47x (  9.4%) ████
  Feature 176:  47x (  9.4%) ████
  Feature 174:  47x (  9.4%) ████
  Feature 173:  47x (  9.4%) ████
  Feature 181:  46x (  9.2%) ████
  Feature 180:  46x (  9.2%) ████
  Feature 178:  46x (  9.2%) ████
  Feature 175:  46x (  9.2%) ████

💾 Saved to: ../results/day2_explanations_v2.pkl (107.8 KB)

SAMPLE EXPLANATIONS (Top 3 Nodes)

[1] Node 339576533 | Anomaly Score: 0.8104
    Top 5 Features:
      • Feature 182: importance=0.005464, value=2.9285
      • Feature 181: importance=0.005464, value=9.1049
      • Feature 180: importance=0.005464, value=7.5134
      • Feature 179: importance=0.005464, value=3.3861
      • Feature 17